# HateBERT Fine-tuning for PCL Detection
Fine-tunes HateBERT with weighted loss to handle class imbalance in the Don't Patronize Me! dataset.

## 1. Imports

In [15]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
import ast
import os

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 2. Config

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR       = 'datasets'
TSV_PATH       = os.path.join(DATA_DIR, 'dontpatronizeme_pcl.tsv')
TRAIN_PATH     = os.path.join(DATA_DIR, 'train_semeval_parids-labels.csv')
DEV_PATH       = os.path.join(DATA_DIR, 'dev_semeval_parids-labels.csv')

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME     = 'GroNLP/hateBERT'
MAX_LEN        = 128     # covers >95% of paragraphs based on EDA (mean=48, median=42)
BATCH_SIZE     = 32
EPOCHS         = 10
LEARNING_RATE  = 1e-5
WARMUP_RATIO   = 0.1     # 10% of steps used for warmup

## 3. Load & Prepare Data

In [3]:
# Load main tsv
df = pd.read_csv(
    TSV_PATH,
    sep='\t',
    header=None,
    names=['par_id', 'art_id', 'keyword', 'country', 'text', 'label'],
    quoting=3,
    engine='python',
    skiprows=4
)

# Binary label: {0,1} -> 0 (No PCL), {2,3,4} -> 1 (PCL)
df['binary_label'] = df['label'].apply(lambda x: 0 if x in [0, 1] else 1)

# Load split files — we only need par_id from these
train_ids = pd.read_csv(TRAIN_PATH)['par_id'].astype(int).tolist()
dev_ids   = pd.read_csv(DEV_PATH)['par_id'].astype(int).tolist()

#Make sure par_id from tsv is also int
df['par_id'] = df['par_id'].astype(int)

# Filter to train and dev splits
train_df = df[df['par_id'].isin(train_ids)].reset_index(drop=True)
dev_df   = df[df['par_id'].isin(dev_ids)].reset_index(drop=True)

print(f'Train size : {len(train_df)} | PCL: {train_df["binary_label"].sum()} ({100*train_df["binary_label"].mean():.1f}%)')
print(f'Dev size   : {len(dev_df)}   | PCL: {dev_df["binary_label"].sum()} ({100*dev_df["binary_label"].mean():.1f}%)')

Train size : 8375 | PCL: 794 (9.5%)
Dev size   : 2094   | PCL: 199 (9.5%)


## 4. Compute Class Weights

In [4]:
# Weight each class inversely proportional to its frequency
n_samples  = len(train_df)
n_pcl      = train_df['binary_label'].sum()
n_no_pcl   = n_samples - n_pcl

weight_no_pcl = n_samples / (2 * n_no_pcl)
weight_pcl    = n_samples / (2 * n_pcl)

class_weights = torch.tensor([weight_no_pcl, weight_pcl], dtype=torch.float).to(device)
print(f'Class weights -> No PCL: {weight_no_pcl:.4f} | PCL: {weight_pcl:.4f}')

Class weights -> No PCL: 0.5524 | PCL: 5.2739


## 5. Dataset & DataLoader

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = PCLDataset(train_df['text'].tolist(), train_df['binary_label'].tolist(), tokenizer, MAX_LEN)
dev_dataset   = PCLDataset(dev_df['text'].tolist(),   dev_df['binary_label'].tolist(),   tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)}')

Train batches: 262 | Dev batches: 66


## 6. Model, Optimiser & Scheduler

In [6]:
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# torch.save(model.state_dict(), 'hatebert_pretrained.pt') #Reproducibility

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.load_state_dict(torch.load('hatebert_pretrained.pt', map_location=device))
model = model.to(device)

# Weighted cross-entropy loss — penalises misclassifying PCL more heavily
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

optimiser = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler     = get_linear_schedule_with_warmup(optimiser, warmup_steps, total_steps)

print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total steps: 2620 | Warmup steps: 262


## 7. Training Loop

In [8]:
def train_epoch(model, loader, optimiser, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimiser.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimiser.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
            loss       = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, pos_label=1)
    return avg_loss, f1, all_preds, all_labels

In [10]:
best_f1        = 0
best_model_path = 'best_hatebert_pcl.pt'

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_epoch(model, train_loader, optimiser, scheduler, loss_fn, device)
    dev_loss, dev_f1, _, _ = evaluate(model, dev_loader, loss_fn, device)

    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'  Train -> Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Dev   -> Loss: {dev_loss:.4f}   | F1: {dev_f1:.4f}')

    # Save best model based on dev F1
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model.state_dict(), best_model_path)
        print(f'  >> New best model saved (F1: {best_f1:.4f})')

print(f'\nBest dev F1: {best_f1:.4f}')

Epoch 1/10
  Train -> Loss: 0.6379 | F1: 0.2396
  Dev   -> Loss: 0.3897   | F1: 0.3981
  >> New best model saved (F1: 0.3981)
Epoch 2/10
  Train -> Loss: 0.4545 | F1: 0.4450
  Dev   -> Loss: 0.3410   | F1: 0.4734
  >> New best model saved (F1: 0.4734)
Epoch 3/10
  Train -> Loss: 0.3242 | F1: 0.6042
  Dev   -> Loss: 0.3812   | F1: 0.5065
  >> New best model saved (F1: 0.5065)
Epoch 4/10
  Train -> Loss: 0.2288 | F1: 0.7368
  Dev   -> Loss: 0.3895   | F1: 0.5179
  >> New best model saved (F1: 0.5179)
Epoch 5/10
  Train -> Loss: 0.1530 | F1: 0.8360
  Dev   -> Loss: 0.4453   | F1: 0.5560
  >> New best model saved (F1: 0.5560)
Epoch 6/10
  Train -> Loss: 0.1024 | F1: 0.9028
  Dev   -> Loss: 0.5585   | F1: 0.5170
Epoch 7/10
  Train -> Loss: 0.0639 | F1: 0.9454
  Dev   -> Loss: 0.5894   | F1: 0.5472
Epoch 8/10
  Train -> Loss: 0.0439 | F1: 0.9594
  Dev   -> Loss: 0.7098   | F1: 0.5095
Epoch 9/10
  Train -> Loss: 0.0311 | F1: 0.9733
  Dev   -> Loss: 0.6846   | F1: 0.5158
Epoch 10/10
  Train ->

## 8. Final Evaluation on Dev Set

In [12]:
# Load best checkpoint and evaluate
model.load_state_dict(torch.load(best_model_path, map_location=device))
_, dev_f1, preds, labels = evaluate(model, dev_loader, loss_fn, device)

print('=== Final Dev Set Results with weighted loss ===')
print(f'F1 (PCL class): {dev_f1:.4f}')
print()
print(classification_report(labels, preds, target_names=['No PCL', 'PCL']))

=== Final Dev Set Results with weighted loss ===
F1 (PCL class): 0.5560

              precision    recall  f1-score   support

      No PCL       0.96      0.93      0.94      1895
         PCL       0.49      0.65      0.56       199

    accuracy                           0.90      2094
   macro avg       0.72      0.79      0.75      2094
weighted avg       0.92      0.90      0.91      2094



## 9. HateBERT with no weighted loss (ablation)

In [11]:

# Reinitialise model and optimiser from scratch
model_ablation = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_ablation.load_state_dict(torch.load('hatebert_pretrained.pt', map_location=device))
model_ablation = model_ablation.to(device)

# Standard cross-entropy — no class weights
loss_fn_ablation = nn.CrossEntropyLoss()

optimiser_ablation = AdamW(model_ablation.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler_ablation = get_linear_schedule_with_warmup(optimiser_ablation, warmup_steps, total_steps)

best_f1_ablation = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1 = train_epoch(model_ablation, train_loader, optimiser_ablation, scheduler_ablation, loss_fn_ablation, device)
    dev_loss, dev_f1, _, _ = evaluate(model_ablation, dev_loader, loss_fn_ablation, device)

    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'  Train -> Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Dev   -> Loss: {dev_loss:.4f}   | F1: {dev_f1:.4f}')

    if dev_f1 > best_f1_ablation:
        best_f1_ablation = dev_f1
        torch.save(model_ablation.state_dict(), 'best_hatebert_no_weights.pt')
        print(f'  >> New best model saved (F1: {best_f1_ablation:.4f})')

print(f'\nBest dev F1 (no weighted loss): {best_f1_ablation:.4f}')

# Final evaluation
model_ablation.load_state_dict(torch.load('best_hatebert_no_weights.pt', map_location=device))
_, dev_f1_ablation, preds_ablation, labels_ablation = evaluate(model_ablation, dev_loader, loss_fn_ablation, device)

print('\n=== Ablation: HateBERT without weighted loss ===')
print(f'F1 (PCL class): {dev_f1_ablation:.4f}')
print()
print(classification_report(labels_ablation, preds_ablation, target_names=['No PCL', 'PCL']))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10
  Train -> Loss: 0.3849 | F1: 0.0735
  Dev   -> Loss: 0.2403   | F1: 0.1121
  >> New best model saved (F1: 0.1121)
Epoch 2/10
  Train -> Loss: 0.2213 | F1: 0.3515
  Dev   -> Loss: 0.2082   | F1: 0.4409
  >> New best model saved (F1: 0.4409)
Epoch 3/10
  Train -> Loss: 0.1629 | F1: 0.6051
  Dev   -> Loss: 0.2142   | F1: 0.5014
  >> New best model saved (F1: 0.5014)
Epoch 4/10
  Train -> Loss: 0.1133 | F1: 0.7642
  Dev   -> Loss: 0.2752   | F1: 0.4247
Epoch 5/10
  Train -> Loss: 0.0715 | F1: 0.8646
  Dev   -> Loss: 0.3314   | F1: 0.4679
Epoch 6/10
  Train -> Loss: 0.0422 | F1: 0.9293
  Dev   -> Loss: 0.4104   | F1: 0.4713
Epoch 7/10
  Train -> Loss: 0.0259 | F1: 0.9631
  Dev   -> Loss: 0.4471   | F1: 0.5211
  >> New best model saved (F1: 0.5211)
Epoch 8/10
  Train -> Loss: 0.0216 | F1: 0.9664
  Dev   -> Loss: 0.4780   | F1: 0.4773
Epoch 9/10
  Train -> Loss: 0.0104 | F1: 0.9879
  Dev   -> Loss: 0.4904   | F1: 0.5205
Epoch 10/10
  Train -> Loss: 0.0092 | F1: 0.9905
  Dev   -> L